In [0]:
# 1 . Bussiness requirements:  A retail company receive a customers.csv every day . your task is to load it into databricks after basic cleaning.

# 1. read csv file.


path = '/FileStore/tables/customer_practice.csv'

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true")\
    .load(path, mode = "overwrite")   

   
df.show()
    


+-------+---------+----------+----+
|cust_id|cust_name|      city| age|
+-------+---------+----------+----+
|    101|     Amit|      Pune|  28|
|    102|    Sneha|    Mumbai|  32|
|    103|    Rahul|    Nagpur|  25|
|    102|    Sneha|    Mumbai|  32|
|    104|    Priya|    Nashik|  30|
|    105|     NULL|      Pune|  27|
|    106|    Karan|      NULL|  35|
|    107|     Neha|Aurangabad|NULL|
|    103|    Rahul|    Nagpur|  25|
+-------+---------+----------+----+



In [0]:
# 2. Remove duplicates records.

from pyspark.sql.functions import *

rd = df.dropDuplicates() 
rd.display()

cust_id,cust_name,city,age
105,null,Pune,27
104,Priya,Nashik,30
103,Rahul,Nagpur,25
107,Neha,Aurangabad,null
106,Karan,null,35
101,Amit,Pune,28
102,Sneha,Mumbai,32


In [0]:
# 3. Remove Row where customer Name is Null.

from pyspark.sql.functions import *

rd = df.na.drop(subset =["cust_name"])
rd.display()

cust_id,cust_name,city,age
101,Amit,Pune,28
102,Sneha,Mumbai,32
103,Rahul,Nagpur,25
102,Sneha,Mumbai,32
104,Priya,Nashik,30
106,Karan,null,35
107,Neha,Aurangabad,null
103,Rahul,Nagpur,25


In [0]:
# 3. Replace NULL value in city, age with 0.

from pyspark.sql.functions import *
rn = rd.na.fill(0, subset= ["age"])
rn1 = rn.na.fill("-", subset =["city"])

rn1.display()



cust_id,cust_name,city,age
101,Amit,Pune,28
102,Sneha,Mumbai,32
103,Rahul,Nagpur,25
102,Sneha,Mumbai,32
104,Priya,Nashik,30
106,Karan,-,35
107,Neha,Aurangabad,0
103,Rahul,Nagpur,25


In [0]:
# 5. Add a load_date column with the current date.

from pyspark.sql.functions import *

ld = rn1.withColumn("load_date", current_date())
ld.display()


cust_id,cust_name,city,age,load_date
101,Amit,Pune,28,2026-07-10
102,Sneha,Mumbai,32,2026-07-10
103,Rahul,Nagpur,25,2026-07-10
102,Sneha,Mumbai,32,2026-07-10
104,Priya,Nashik,30,2026-07-10
106,Karan,-,35,2026-07-10
107,Neha,Aurangabad,0,2026-07-10
103,Rahul,Nagpur,25,2026-07-10


In [0]:
# 6. Save cleaned data as Delta table named silver_customer.

ld.write.format("delta").mode("overwrite").saveAsTable("silver_customer")
spark.table("silver_customer").display()




cust_id,cust_name,city,age,load_date
101,Amit,Pune,28,2026-07-10
102,Sneha,Mumbai,32,2026-07-10
103,Rahul,Nagpur,25,2026-07-10
102,Sneha,Mumbai,32,2026-07-10
104,Priya,Nashik,30,2026-07-10
106,Karan,-,35,2026-07-10
107,Neha,Aurangabad,0,2026-07-10
103,Rahul,Nagpur,25,2026-07-10
